# Constrain piece-wise constant pulses to vary smoothly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.measurement.smoothness import Smoothness
from paraqeet.optimisation_map import OptimisationMap
from paraqeet.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient

In [ ]:
delta_sampling = 33e-9
n_pwc = 40  # number of piecewise constants in the pulse
t_final = n_pwc * delta_sampling  # for now just set up for trying
tlist = np.linspace(0, t_final, n_pwc + 1)
eps_qubit = 2 * np.pi * 1.0  # initial amplitude of the qubit (in MHz)
eps_max_qubit = 5 * eps_qubit  # maximum amplitude of the resonator (in MHz)
tone_qubit = GaussEnvelope(
    amplitude=Quantity(eps_qubit * 1e6, -eps_max_qubit * 1e6, eps_max_qubit * 1e6),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final),
)
gen_qubit = PWCGenerator(envelopes=[tone_qubit], max_amplitude=eps_max_qubit * 1e6, tlist=tlist)

In [ ]:
from paraqeet.plotting import plot_signal

ts = np.linspace(0, t_final, 501)
fig, ax = plt.subplots(1, figsize=(5, 3))
plot_signal(tone_qubit, ts, ax, linestyle="-", label="Smooth")
plot_signal(gen_qubit, ts, ax, linestyle="--", label="PWC")
ax.legend(loc=1, frameon=True)
plt.show()

In [ ]:
optmap = OptimisationMap()
optmap.add(gen_qubit, gen_qubit.get_parameters())

# We add dummy parameters to check if the gradient is computed
# correctly by padding zeros
# optmap.add(tone_qubit, tone_qubit.get_parameters())

optmap.register_params_with_optimisables()

In [ ]:
smoothness = Smoothness(pwc_generator=gen_qubit)

We can check that the gradient has the correct shape

In [ ]:
print(smoothness.measure_with_gradient()[1].shape)

In [ ]:
opt = ScipyOptimiserGradient(smoothness, optimisation_map=optmap)
max_iter = 200
opt.set_options({"maxiter": max_iter})

In [ ]:
opt.optimise()

In [ ]:
smoothness.measure()

In [ ]:
smoothness.measure_with_gradient()

In [ ]:
plot_signal(gen_qubit, ts, linestyle="--", label="PWC");

As expected obtain a flat pulse.